# TFT Data Exploration — UCI Electricity Dataset (Cycle 2)

This notebook validates the `ElectricityDataModule` output using the UCI Electricity Load Diagrams 2011-2014 dataset (370 entities):
- Batch shapes match TFT input requirements
- Value ranges are reasonable (scaled data)
- Temporal features and holiday flag are correctly encoded
- No data leakage across splits

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import matplotlib.pyplot as plt
from src.data import ElectricityDataModule, load_electricity_data

## 1. Initialize Data Module and Load a Batch

In [ ]:
# Use UCI Electricity data with 5 entities for fast notebook execution
dm = ElectricityDataModule(
    max_entities=5,
    lookback=168,
    horizon=24,
    batch_size=64,
    use_uci=True,
)
dm.setup()
print('Split info:', dm.split_info)
print('Date boundaries:', dm.date_boundaries)

past, future, static, targets = dm.get_sample_batch()
print(f'\npast_inputs shape:         {past.shape}')
print(f'known_future_inputs shape: {future.shape}')
print(f'static_inputs shape:       {static.shape}')
print(f'targets shape:             {targets.shape}')

## 2. Shape Verification

Expected shapes (UCI Electricity):
- `past_inputs`: (batch_size, lookback=168, n_past_features=5) — 1 observed + 4 temporal
- `known_future_inputs`: (batch_size, horizon=24, n_known_features=5) — 4 temporal + 1 holiday
- `static_inputs`: (batch_size, 1) — entity ID
- `targets`: (batch_size, horizon=24)

In [ ]:
assert past.shape == (64, 168, 5), f'Unexpected past shape: {past.shape}'
assert future.shape == (64, 24, 5), f'Unexpected future shape: {future.shape}'
assert static.shape == (64, 1), f'Unexpected static shape: {static.shape}'
assert targets.shape == (64, 24), f'Unexpected targets shape: {targets.shape}'
print('All shape assertions passed!')

## 3. Value Range Analysis

In [ ]:
feature_names_past = ['power_usage (scaled)',
                      'hour_of_day', 'day_of_week', 'month', 'day_of_month']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, name) in enumerate(zip(axes.flat[:5], feature_names_past)):
    vals = past[:, :, i].numpy().flatten()
    ax.hist(vals, bins=50, alpha=0.7)
    ax.set_title(f'{name}\nrange: [{vals.min():.2f}, {vals.max():.2f}]')
    ax.set_xlabel('Value')
axes.flat[5].set_visible(False)
plt.suptitle('Past Input Feature Distributions — UCI Electricity (1 batch)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/cycle_2/past_features_dist.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Sample Sequence Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

for sample_idx in range(3):
    ax = axes[sample_idx]
    eid = static[sample_idx, 0].item()
    
    # Past power_usage values (first observed feature)
    past_power = past[sample_idx, :, 0].numpy()
    # Target values
    tgt = targets[sample_idx].numpy()
    
    x_past = np.arange(len(past_power))
    x_future = np.arange(len(past_power), len(past_power) + len(tgt))
    
    ax.plot(x_past, past_power, 'b-', label='Past (lookback)', alpha=0.8)
    ax.plot(x_future, tgt, 'r-', label='Target (horizon)', linewidth=2)
    ax.axvline(x=len(past_power), color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'Sample {sample_idx} — Customer MT_{eid+1:03d}')
    ax.legend()
    ax.set_ylabel('Scaled power usage')

axes[-1].set_xlabel('Time step (hours)')
plt.suptitle('Lookback + Horizon Windows — UCI Electricity (3 samples)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/cycle_2/sample_sequences.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Known Future Inputs (Temporal Features + Holiday Flag)

In [ ]:
known_future_names = ['hour_of_day', 'day_of_week', 'month', 'day_of_month', 'is_holiday']

fig, axes = plt.subplots(1, 5, figsize=(20, 3))
sample = future[0].numpy()  # (24, 5)
for i, (ax, name) in enumerate(zip(axes, known_future_names)):
    ax.plot(sample[:, i], 'o-', markersize=3)
    ax.set_title(name)
    ax.set_xlabel('Horizon step')
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Known Future Inputs for 1 Sample (24h horizon) — includes holiday flag', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/cycle_2/future_inputs.png', dpi=100, bbox_inches='tight')
plt.show()

# Holiday statistics
all_holidays = dm.train_dataset.known_future_inputs[:, :, 4]
pct = (all_holidays == 1.0).float().mean().item() * 100
print(f'Holiday percentage in training data: {pct:.2f}%')

## 6. Data Split Summary & Leakage Check

In [ ]:
print('Dataset sizes:')
print(f'  Train: {len(dm.train_dataset)} samples')
print(f'  Val:   {len(dm.val_dataset)} samples')
print(f'  Test:  {len(dm.test_dataset)} samples')
print(f'  Total: {len(dm.train_dataset) + len(dm.val_dataset) + len(dm.test_dataset)} samples')
print(f'\nData source: {dm.split_info["data_source"]}')
print(f'Entities: {dm.split_info["n_entities"]}')
print(f'Lookback: {dm.split_info["lookback"]} hours')
print(f'Horizon:  {dm.split_info["horizon"]} hours')
print(f'Past features: {dm.split_info["n_past_features"]}')
print(f'Known future features: {dm.split_info["n_known_future_features"]}')

# Verify no NaN values
for name, ds in [('train', dm.train_dataset), ('val', dm.val_dataset), ('test', dm.test_dataset)]:
    has_nan = (
        ds.past_inputs.isnan().any().item() or
        ds.known_future_inputs.isnan().any().item() or
        ds.targets.isnan().any().item()
    )
    print(f'{name}: NaN present = {has_nan}')

print('\nAll checks passed!')